In [21]:
import pinecone
from sentence_transformers import SentenceTransformer

In [22]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [23]:
import time

pc = pinecone.Pinecone(api_key="4ddacf03-01b2-4d67-9c8d-860e9e0ff057", environment="gcp-starter")
index_name = 'query-function-mapping'

# if index_name in pc.list_indexes():
#     pc.delete_index(index_name)
    
# pinecone.create_index(
#     name=index_name,
#     dimension=384,
#     metric='cosine'
# )
# # wait for index to finish initialization
# while not pinecone.describe_index(index_name).status['ready']:
#     time.sleep(1)

# Connect to your Pinecone index
index = pc.Index(index_name)

In [24]:
def generate_embedding(text):
    return model.encode(text).tolist()

In [25]:
queries_functions = [
    # ("Please analyze historical execution data to measure the quality of past trades in terms of slippage, price deviation, and trade completeness", "execution_quality_analysis"),
    # ("Evaluate historical performance with different counterparties, considering factors like price improvement, execution speed, and reliability", "counterparty_performance"),
    # ("Conduct Transaction Cost Analysis (TCA) on historical trades to understand transaction costs, including market impact, spread costs, and slippage", "transaction_cost_analysis"),
    # ("Analyze historical trades against VWAP to assess the effectiveness of execution relative to the market.", "volume_weighted_average_price"),
    # ("Examine how the liquidity profile of specific securities has changed over time", "liquidity_profile_over_time"),
    # ("Review the historical impact of different trade sizes on execution performance", "trade_size_optimization"),
    # ("Analyze post-trade execution results based on different asset classes and market segments", "post_trade_analysis_by_asset_class"),
    # ("Assess the historical impact of the duration of trades on execution quality", "duration_analysis"),
    # ("Calculate risk-adjusted performance metrics for historical trades", "risk_adjusted_metrics"),


    ("Analyze my 10Y Apple bond trades against similar securities on TRACE over the last week. Please give the results in terms of spread to treasury (STT)", "analyze_similar_trades"),
    ("Benchmark historical trade performance against market trade data / TRACE data for similar fixed income securities", "trade_vs_trace"),
    ("Conduct TCA on historical trades to understand transaction costs, including price impact, price improvement, and slippage", "calculate_tca_metrics"),
    ("Analyze historical treasury trades against VWAP on the NYSE to assess the effectiveness of execution relative to the market", "analyze_trades_vwap"),
    ("Examine how the liquidity profile of specific securities in the market has changed over time", "liquidity_over_time"),
    ("Review the historical impact of different trade sizes on execution performance", "trade_size_vs_execution_performance"),
    ("Analyze post-trade execution results based on different fixed income asset classes", "post_trade_analytics"),
    ("Calculate risk-adjusted performance metrics for historical trades", "risk_adjusted_performance_metrics"),
    ("Please analyze historical execution data to measure the quality of past trades in terms of slippage, price deviation, and trade completeness", "calculate_slippage"),
    ("Evaluate historical performance with different counterparties, considering factors like price improvement, execution speed, and reliability", "counterparty_performance"),

    # # charts
    ("Could you plot a line chart showing the spread to treasury (STT) between our trades and similar market securities for the last week? Analyze the chart data briefly.", "trade_vs_similar_securities"),
    ("Could you plot a line chart showing the comparison between our trade prices and VWAP for the first 10 trading days?", "trade_prices_vs_vwap"),
    ("Can you give me a chart showing how price impact and trade completeness vary with trade size?", "price_impact_trade_completeness"),
    ("Could you provide a visualization comparing price impact and trade completeness across these fixed income asset classes?", "post_trade_by_class"),
    ("Can you provide a chart showing price impact, price improvement and slippage metrics for the first 15 trades?", "tca_metrics_chart"),
    ("Can you plot a chart showing the bid-ask spread over the last month?", "bid_ask_spread"),
    ("Can you show me a bar chart comparing our trade yields to the average TRACE yields over the last month?", "trade_vs_trace_chart"),
    ("Can you give me a graph for past trades in terms of slippage, price deviation, and trade completeness?", "calculate_slippage_chart"),
]

for query, function_name in queries_functions:
    embedding = generate_embedding(query)
    print(embedding)
    metadata = {"description": query}
    print(f"Inserting {function_name} with metadata {metadata}")
    index.upsert(vectors=[(function_name, embedding, metadata)])

print("Queries and functions have been inserted into Pinecone.")

[-0.011587586253881454, -0.014724656008183956, -0.03755512461066246, -0.060563825070858, -0.016877733170986176, -0.024301869794726372, 0.012693415395915508, 0.10642046481370926, -0.06266414374113083, -0.03421914577484131, 0.06363901495933533, 0.026224685832858086, -0.0420975536108017, 0.010187829844653606, -0.017466416582465172, -0.03500987961888313, -0.04742836579680443, 0.010897094383835793, -0.08010463416576385, 0.0138009088113904, 0.0077114771120250225, 0.011164502240717411, -0.03493690863251686, -0.048777755349874496, 0.05780540034174919, 0.02224179171025753, -0.002585415495559573, -0.03478216752409935, -0.04528786242008209, -0.03187473118305206, -0.01603783294558525, -0.042547598481178284, -0.004112333990633488, 0.10219358652830124, -0.011330953799188137, -0.047496698796749115, -0.018539009615778923, -0.05774109065532684, 0.04125722870230675, -0.03862448036670685, -0.002873287070542574, 0.011627727188169956, 0.03376397863030434, 0.011398636735975742, -0.04190598055720329, -0.0759

In [13]:
def search_closest_function(user_query):
    # Generate embedding for the user query
    user_embedding = generate_embedding(user_query)

    # Search in Pinecone for the closest match
    search_result = index.query(vector=user_embedding, top_k=1)

    # Retrieve the most similar function name and its similarity score
    if search_result['matches']:
        match = search_result['matches'][0]
        function_name = match['id']
        similarity_score = match['score']
        print(
            f"Closest match found: {function_name} with a similarity score of {similarity_score}")
        return function_name, similarity_score
    else:
        print("No match found.")
        return None, None

In [14]:
user_query = "Calculate risk-adjusted performance metrics for historical trades"

# Search for the closest matching predefined query and associated function
closest_function, _ = search_closest_function(user_query)

if closest_function:
    print(f"Function to call: {closest_function}")

No match found.
